# 5) SVM (Support Vector Machine) Nedir?

SVM, sınıflandırma problemine tamamen geometrik bir açıdan yaklaşır "en iyi ayırıcı çizgiyi/düzlemi nasıl çizerim" sorusuna odaklanır.

### Temel Mantık: Margin Maximization

İki sınıfı ayıran sonsuz sayıda çizgi çizilebilir, hepsi de sınıfları doğru ayırabilir. SVM, bunlardan en iyisini şu şekilde seçer: "İki sınıfa da en uzak olan çizgiyi çiz." Bu, çizginin her iki taraftaki en yakın noktalara olan mesafesini maksimize etmek demektir. Bu mesafeye margin (kenar boşluğu) denir.

### Support Vectors Nedir?

Çizgiye en yakın olan, marjini belirleyen noktalara "support vectors" (destek vektörleri) denir, modelin ismi de buradan gelir. SVM, karar sınırını belirlerken sadece bu sınır noktalarına bakar, uzaktaki (net bir şekilde bir sınıfa ait olan) noktalar sonucu etkilemez.

### Aykırı Değerlere Karşı Hassasiyet

SVM, uzaktaki "normal" noktaları görmezden gelir ama aykırı değerler (outlier) genelde kendi sınıfının normal bölgesinden uzak, diğer sınıfa yakın bir yerde durur bu da onları tam olarak "sınıra yakın" bir nokta, yani potansiyel bir support vector yapar. Bu yüzden tek bir aykırı değer, marjini ve dolayısıyla tüm karar sınırını önemli ölçüde kaydırabilir. SVM bu açıdan aykırı değerlere karşı hassastır.

### Soft Margin ve C Parametresi

Bu hassasiyeti dengelemek için "soft margin" (esnek kenar boşluğu) tekniği kullanılır. Model, bazı noktaları (özellikle aykırı olanları) yanlış sınıflandırmayı göze alarak, genel düzeni koruyabilir.

Bu denge, C parametresi ile kontrol edilir:
- C büyük → Model "katı" davranır, her noktayı (aykırı olsa bile) doğru sınıflandırmaya zorlanır → aykırı değerlere karşı hassas, overfitting riski
- C küçük → Model "esnek" davranır, birkaç noktayı yanlış sınıflandırmayı göze alır, genel yapıyı yakalamaya odaklanır → aykırı değerlere karşı dayanıklı, ama çok küçükse underfitting riski

### Kernel Trick (İleri Konu)

Bazen veriler düz bir çizgiyle ayrılamayacak kadar karmaşık dağılmış olabilir (örneğin iç içe geçmiş iki daire gibi). SVM'in "kernel trick" denen bir tekniği, veriyi daha yüksek boyutlu bir uzaya taşıyarak orada düz bir çizgiyle ayrılabilir hale getirir bu, Polynomial Regression'da x² eklememize benzer bir mantık taşır; veriye yeni bir bakış açısı kazandırılır.

In [2]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --- Veri Yükleme ve Temizleme ---
titanic = sns.load_dataset('titanic')
titanic['age'] = titanic.groupby(['pclass', 'sex'])['age'].transform(lambda x: x.fillna(x.median()))
titanic = titanic.drop(columns=['deck'])
titanic = titanic.dropna(subset=['embark_town'])
titanic = titanic.drop(columns=['alive', 'embarked'])
titanic = pd.get_dummies(titanic, columns=['embark_town', 'sex'], drop_first=True)
titanic['adult_male'] = titanic['adult_male'].astype(int)
sinif_siralamasi = {'First': 1, 'Second': 2, 'Third': 3}
titanic['class'] = titanic['class'].map(sinif_siralamasi)
titanic = titanic.drop(columns=['who', 'class'])
bool_sutunlar = titanic.select_dtypes(include='bool').columns
titanic[bool_sutunlar] = titanic[bool_sutunlar].astype(int)

# --- Model İçin Hazırlık ---
X = titanic.drop(columns=['survived'])
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SVM da mesafe tabanlı çalıştığı için (KNN gibi) Scaling şart
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Farklı C Değerleriyle Model Karşılaştırması ---
for c_deger in [0.01, 0.1, 1, 10, 100]:
    svm_model = SVC(C=c_deger, kernel='linear')
    svm_model.fit(X_train_scaled, y_train)
    y_pred = svm_model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"C={c_deger}: Accuracy={acc:.4f}")
# En iyi C değerini yukarıdaki sonuçlara göre seç ve detaylı incele
en_iyi_c = 1  # örnek, sen sonuçlara göre güncelle
svm_final = SVC(C=en_iyi_c, kernel='linear')
svm_final.fit(X_train_scaled, y_train)
y_pred_final = svm_final.predict(X_test_scaled)

print(f"\nEn iyi C={en_iyi_c} ile:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

C=0.01: Accuracy=0.8034
C=0.1: Accuracy=0.8146
C=1: Accuracy=0.8146
C=10: Accuracy=0.8146
C=100: Accuracy=0.8146

En iyi C=1 ile:
Accuracy: 0.8146

Confusion Matrix:
[[91 18]
 [15 54]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.85       109
           1       0.75      0.78      0.77        69

    accuracy                           0.81       178
   macro avg       0.80      0.81      0.81       178
weighted avg       0.82      0.81      0.82       178



### SVM Sonuçları ve Genel Model Karşılaştırması

C değeri arttıkça (0.01→0.1) accuracy hafif yükseldi (0.8034→0.8146) ama C=0.1'den sonra (1, 10, 100) hiç değişmedi — bu, veri setinin doğrusal olarak zaten stabil bir şekilde ayrılabildiğini, modeli daha "katı" yapmanın ekstra fayda sağlamadığını gösteriyor.

**4 Model Karşılaştırması:**
- Logistic Regression: 0.8258 (en iyi)
- KNN (K=15): 0.8146
- Naive Bayes: 0.8146
- SVM (C≥0.1): 0.8146

Üç farklı algoritma (KNN, Naive Bayes, SVM) aynı accuracy'de buluştu, Logistic Regression hepsinden hafif önde. Bu, Titanic veri seti için doğrusal bir karar sınırının yeterince iyi çalıştığını, algoritma seçiminden çok veri hazırlama/feature engineering'in belirleyici olduğunu gösteriyor.